### Προετοιμασία δεδομένων

Το πρώτο κελί προετοιμάζει αυτόματα τα εκπαιδευτικά δεδομένα. Εκτέλεσέ το πριν ξεκινήσεις την άσκηση.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import json

DATA_ROOT = next((p for p in (Path("../../data"), Path("data"), Path("downloads/python/data")) if p.exists()), Path("data"))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
DATA_URLS = {
    "observations_dirty.csv": "https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp02/observations_dirty.csv"
}

def valid_cached(path):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        if path.suffix.lower() == ".gpkg":
            return path.read_bytes()[:16] == b"SQLite format 3\x00"
        if path.suffix.lower() == ".geojson":
            json.loads(path.read_text(encoding="utf-8"))
        return True
    except Exception:
        return False

for filename, url in DATA_URLS.items():
    target = DATA_ROOT / "gp02" / filename
    if valid_cached(target):
        print(f"[OK] Ήδη διαθέσιμο: {target}")
        continue
    target.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urlopen(url, timeout=30) as response:
            payload = response.read()
        if not payload:
            raise RuntimeError("κενή απάντηση")
        target.write_bytes(payload)
        print(f"[OK] Κατέβηκε: {target}")
    except Exception as exc:
        if target.exists():
            target.unlink()
        print(f"[WARN] Δεν ήταν δυνατή η λήψη {url}: {exc}")


# GP02 — Clean Geospatial Data

Σε αυτό το notebook θα καθαρίσεις ένα σκόπιμα προβληματικό dataset. Η αρχή είναι: εντοπίζω → κατανοώ → αποφασίζω → τεκμηριώνω → ελέγχω ξανά. Μην διαγράψεις γραμμές πριν μετρήσεις τι χάνεται.

## 1. Load — Φόρτωση και πρώτη επιθεώρηση

Πρώτα μετράμε πριν διορθώσουμε.

In [ ]:
import pandas as pd
import geopandas as gpd
df = pd.read_csv(DATA_ROOT / "gp02/observations_dirty.csv")
print("Rows:", len(df))
print(df.dtypes)
print(df.isna().sum())


## 2. Missing values και data types

Ξεχώρισε essential fields (`observation_id`, longitude, latitude) από optional fields. Το `errors="coerce"` κάνει μη αναγνώσιμες τιμές missing· μετά πρέπει να μετρήσεις τα νέα missing και να αιτιολογήσεις τον κανόνα.

In [ ]:
# Μετέτρεψε τις τρεις στήλες σε αριθμούς.
# Μέτρησε ξανά τα missing values.
for column in ["longitude", "latitude", "elevation_m"]:
    pass


## 3. Duplicates και συντεταγμένες

Αφαίρεσε γραμμές χωρίς observation_id/coordinates, κράτησε μοναδικά IDs και έλεγξε τα όρια EPSG:4326.

In [ ]:
# Συμπλήρωσε τον καθαρισμό.
# clean = ...
# valid_coordinates = ...


## 4. Geometry

Δημιούργησε Points με longitude ως X και latitude ως Y.

In [ ]:
# gdf = gpd.GeoDataFrame(..., crs="EPSG:4326")
# print(gdf.geometry.geom_type.value_counts())
# print(gdf.total_bounds)


## 5. Before / after QA

In [ ]:
# Πρόσθεσε checks για rows, CRS, missing, empty/invalid geometry, μοναδικά IDs και coordinate ranges.
# Κάνε και έναν γρήγορο plot.


## Mini Challenge

Κράτησε ενεργές παρατηρήσεις με elevation_m > 80 και επανέλαβε τους QA checks.

In [ ]:
# selected = ...
# selected.plot(figsize=(7, 6), color="orange")
